In [4]:
%pip install matplotlib

  Using cached contourpy-1.3.3-cp313-cp313-win_amd64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached kiwisolver-1.4.9-cp313-cp313-win_amd64.whl.metadata (6.4 kB)
   ---------------------------------------- 0.0/8.1 MB ? eta -:--:--
   ---------------------------------------- 8.1/8.1 MB 40.9 MB/s  0:00:00
Using cached contourpy-1.3.3-cp313-cp313-win_amd64.whl (226 kB)
Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
   ---------------------------------------- 0.0/2.3 MB ? eta -:--:--
   ---------------------------------------- 2.3/2.3 MB 32.5 MB/s  0:00:00
Using cached kiwisolver-1.4.9-cp313-cp313-win_amd64.whl (73 kB)

   -------- ------------------------------- 1/5 [fonttools]
   -------- ------------------------------- 1/5 [fonttools]
   -------- ------------------------------- 1/5 [fonttools]
   -------- ------------------------------- 1/5 [fonttools]
   -------- ------------------------------- 1/5 [fonttools]
   -------- -----

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
%pip install pandas pyarrow

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
%pip install seaborn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns


abm = pd.read_parquet("abm.parquet")
card = pd.read_parquet("card.parquet")
cheque = pd.read_parquet("cheque.parquet")
eft = pd.read_parquet("eft.parquet")
emt = pd.read_parquet("emt.parquet")
westernunion = pd.read_parquet("westernunion.parquet")
wire = pd.read_parquet("wire.parquet")

kyc_individual = pd.read_parquet("kyc_individual.parquet")
kyc_smallbusiness = pd.read_parquet("kyc_smallbusiness.parquet")
kyc_occupation_codes = pd.read_parquet("kyc_occupation_codes.parquet")
kyc_industry_codes = pd.read_parquet("kyc_industry_codes.parquet")

labels = pd.read_parquet("labels.parquet")

In [10]:
abm.head()
abm.shape

(185690, 9)

In [9]:
card.head()
card.shape

(3553303, 10)

In [11]:
cheque.head()
cheque.shape

(240548, 5)

In [12]:
eft.head()
eft.shape

(1070698, 5)

In [13]:
emt.head()
emt.shape

(845996, 5)

In [14]:
westernunion.head()
westernunion.shape

(2142, 5)

In [15]:
wire.head()
wire.shape

(4956, 5)

In [16]:
kyc_individual.head(30)
kyc_individual.shape

(53099, 10)

In [17]:
kyc_industry_codes.head(50)
kyc_industry_codes.shape

(168, 2)

In [ ]:
kyc_occupation_codes.head()
kyc_occupation_codes.shape

(97, 2)

In [19]:
kyc_smallbusiness.head()
kyc_smallbusiness.shape

(8311, 9)

In [20]:
labels.head()
labels.shape

(1000, 2)

## Feature Engineering (Tony)

### rapid_outflow_ratio_24h

 = Total debits within 24h of cash deposits/ Total cash deposit amount

 Measures how much of deposited cash leaves the account within 24hrs. Higher ratio indicates the account is being used to quickly pass funds through rather than hold or spend them, which is typical in layering and money rule activity.

### emt_velocity_monthly_mean

= Number of total EMT transactions per month in average

Counts how frequently the client uses email money transfer (EMT) within a one-month period. Unusually high digital transfer frequency suggests rapid movement of funds that may indicate mule networks or account cycling.

### income_transaction_gap_score

Monthly Total Credit = $\sum$(credits across ABM, EMT, EFT, WIRE, CHEQUE)

Declared Monthly Income = $\frac{\text{kyc income}}{12}$

Inme transaction gap score = Monthly Total Credit / Decalred Monthly Income

Comparing actual monthly inflows to the income declared in KYC records. A large gap indicates the client is receiving more money than their reported income supports, which may signal undeclared business activity or use of the account to receive thrid-party funds.

### cash_value_ratio

= $\frac{\text{Total ABM cash transactions}}{\text{Total transatcion amount across all channels}}$

Measures how much of the client's total activity is conducted in physical cash. Excessive reliance on cash to total account activity is inconsistent with most profiles with salaries and increases AML risk due to reduction in track.

### spending_income_ratio

- Monthly card Spend = $\sum$ (CARD debits)

= Monthly Card Spend/ Decalred Monthly Income.

Compares client spending to decalred income. A ratio substantially above 1 suggests the client is spending beyond reported means, which may indicate hidden income sources or financial stress masked by incoming suspicious funds.

### abm_withdrawal_zscore (optional)

 $$= \frac{\text{Client Average Withdrawal} - \mu_{peer}}{\sigma_{peer}}$$

 Within-group abm withdrawal zscore, peer indicates different occupation code or segment

 Identifying clients whose ATM withdrawals are significantly larger than peers with similar profiles. Large and repeated cash withdrawals can indicate cash structuring or conversion of electronic funds into untraceable form.

### small_cash_geo_entropy_30d (optional)

- It filters ABM cash deposits below a proxy threshold.

- Compute proportion of deposits per city:
$p_i = \frac{\text{deposits in city } i}{\text{Total small deposits}}$

- And then the entropy
 = $\sum p_i\log(p_i)$

In [21]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import average_precision_score
from sklearn.model_selection import train_test_split

## Preprocessing

## FE - Feature Engineering codes

### Preprocessing, Aggregating

In [22]:
# 0) Global settings
CHUNK_CARD = 400_000
CHUNK_BIG  = 300_000   # for EFT/EMT
CHUNK_MED  = 200_000   # for cheque
CHUNK_ABM  = 200_000

RANDOM_STATE = 42

# 1) Helpers
def _prep_tx_chunk(chunk: pd.DataFrame) -> pd.DataFrame:
    """Standardize types in a tx chunk."""
    c = chunk.copy()
    c["transaction_datetime"] = pd.to_datetime(c["transaction_datetime"], errors="coerce")
    c["amount_cad"] = pd.to_numeric(c["amount_cad"], errors="coerce")
    c["debit_credit"] = c["debit_credit"].astype(str).str.upper()
    c = c.dropna(subset=["customer_id", "transaction_datetime", "amount_cad", "debit_credit"])
    return c

def _iter_chunks(df: pd.DataFrame, chunksize: int):
    """Yield slices from an in-memory DF."""
    n = len(df)
    for start in range(0, n, chunksize):
        yield df.iloc[start:start+chunksize]

def entropy_from_counts(counts: pd.Series) -> float:
    """Shannon entropy given counts per category for one customer."""
    s = counts.sum()
    if s <= 0:
        return 0.0
    p = counts / s
    # safe log
    return float(-(p * np.log(p)).sum())

def cluster_unique_text(text_series: pd.Series, n_clusters: int) -> np.ndarray:
    """
    Fast clustering:
    - Clean text
    - Vectorize unique strings only
    - MiniBatchKMeans
    """
    txt = (
        text_series.fillna("unknown")
        .astype(str).str.lower()
        .str.replace(r"[^a-z\s]+", " ", regex=True)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

    uniq = txt.drop_duplicates().reset_index(drop=True)
    if len(uniq) <= 3:
        return np.zeros(len(txt), dtype=int)

    vec = TfidfVectorizer(
        stop_words="english",
        max_features=600,     # smaller/faster
        ngram_range=(1, 1),
        min_df=1,
        max_df=0.95
    )
    X = vec.fit_transform(uniq)

    k = min(n_clusters, len(uniq))
    km = MiniBatchKMeans(
        n_clusters=k,
        random_state=RANDOM_STATE,
        batch_size=1024,
        n_init="auto"
    )
    labels = km.fit_predict(X)

    map_df = pd.DataFrame({"text": uniq, "cluster": labels})
    out = txt.to_frame("text").merge(map_df, on="text", how="left")["cluster"].to_numpy()
    return out


# 2) Build KYC master with codebook merges + fast NLP sectors

def build_kyc_master(
    kyc_individual: pd.DataFrame,
    kyc_smallbusiness: pd.DataFrame,
    kyc_occupation_codes: pd.DataFrame,
    kyc_industry_codes: pd.DataFrame
):
    ind = kyc_individual.copy()
    sb  = kyc_smallbusiness.copy()

    # string keys
    ind["occupation_code"] = ind["occupation_code"].astype(str)
    kyc_occupation_codes["occupation_code"] = kyc_occupation_codes["occupation_code"].astype(str)

    sb["industry_code"] = sb["industry_code"].astype(str)
    kyc_industry_codes["industry_code"] = kyc_industry_codes["industry_code"].astype(str)

    # merge titles
    ind = ind.merge(kyc_occupation_codes[["occupation_code", "occupation_title"]],
                    on="occupation_code", how="left")
    sb  = sb.merge(kyc_industry_codes[["industry_code", "industry"]],
                   on="industry_code", how="left")

    # employment flags from occupation_code
    occ_upper = ind["occupation_code"].str.upper()
    ind["is_retired"]    = (occ_upper == "RETIRED").astype(int)
    ind["is_unemployed"] = (occ_upper == "UNEMPLOYED").astype(int)

    # declared monthly income (individuals only)
    ind["declared_monthly_income"] = pd.to_numeric(ind["income"], errors="coerce") / 12.0

    # text fields for clustering
    ind["occupation_text"] = ind["occupation_title"].fillna(ind["occupation_code"])
    sb["industry_text"]    = sb["industry"].fillna(sb["industry_code"])

    # fast sector IDs (codebooks are tiny anyway)
    ind["occupation_sector_id"] = cluster_unique_text(ind["occupation_text"], n_clusters=25)
    sb["industry_sector_id"]    = cluster_unique_text(sb["industry_text"], n_clusters=20)

    # customer universe (for output table)
    cust_ind = ind[["customer_id"]].assign(customer_type="IND")
    cust_sb  = sb[["customer_id"]].assign(customer_type="SB")
    customers = pd.concat([cust_ind, cust_sb], ignore_index=True).drop_duplicates("customer_id")

    return ind, sb, customers

kyc_ind, kyc_sb, customers = build_kyc_master(
    kyc_individual, kyc_smallbusiness, kyc_occupation_codes, kyc_industry_codes
)



## Engineering Features

In [32]:
# 3) Feature: EMT velocity 

def feature_emt_velocity_monthly_mean(emt: pd.DataFrame, chunksize=CHUNK_BIG):

    monthly = {}

    for chunk in _iter_chunks(emt, chunksize):
        c = _prep_tx_chunk(chunk)
        c["month"] = c["transaction_datetime"].dt.to_period("M").astype(str)

        g = c.groupby(["customer_id", "month"]).size()

        for (cust, m), cnt in g.items():
            monthly[(cust, m)] = monthly.get((cust, m), 0) + int(cnt)

    # compute mean per customer
    cust_sum = {}
    cust_n = {}

    for (cust, m), cnt in monthly.items():
        cust_sum[cust] = cust_sum.get(cust, 0) + cnt
        cust_n[cust] = cust_n.get(cust, 0) + 1

    out = pd.DataFrame({
        "customer_id": list(cust_sum.keys()),
        "emt_velocity_monthly_mean": [
            cust_sum[c] / cust_n[c] for c in cust_sum
        ]
    })

    return out

feat_emt_vel = feature_emt_velocity_monthly_mean(emt)

In [30]:
# 4) Feature: Avg monthly credits across ALL channels (for income gap)
#    Chunk-accumulate monthly credit sums across sources, then mean by customer

def accumulate_monthly_credit_sums(monthly_sum_dict: dict, df: pd.DataFrame, chunksize: int):
    for chunk in _iter_chunks(df, chunksize):
        c = _prep_tx_chunk(chunk)
        c = c[c["debit_credit"].eq("C")]  # credits only
        if c.empty:
            continue
        c["month"] = c["transaction_datetime"].dt.to_period("M").astype(str)
        g = c.groupby(["customer_id", "month"])["amount_cad"].sum()
        for (cust, m), amt in g.items():
            monthly_sum_dict[(cust, m)] = monthly_sum_dict.get((cust, m), 0.0) + float(amt)

def monthly_mean_from_dict(monthly_sum_dict: dict, colname: str) -> pd.DataFrame:
    # customer sums and month counts
    cust_sum = {}
    cust_nmo = {}
    for (cust, m), s in monthly_sum_dict.items():
        cust_sum[cust] = cust_sum.get(cust, 0.0) + float(s)
        cust_nmo[cust] = cust_nmo.get(cust, 0) + 1
    out = pd.DataFrame({"customer_id": list(cust_sum.keys())})
    out[colname] = [cust_sum[c]/cust_nmo[c] for c in out["customer_id"]]
    return out

monthly_credit = {}
# Note: ABM has minute-granularity; still fine
accumulate_monthly_credit_sums(monthly_credit, abm, chunksize=CHUNK_ABM)
accumulate_monthly_credit_sums(monthly_credit, card, chunksize=CHUNK_CARD)
accumulate_monthly_credit_sums(monthly_credit, cheque, chunksize=CHUNK_MED)
accumulate_monthly_credit_sums(monthly_credit, eft, chunksize=CHUNK_BIG)
accumulate_monthly_credit_sums(monthly_credit, emt, chunksize=CHUNK_BIG)
accumulate_monthly_credit_sums(monthly_credit, westernunion, chunksize=50_000)
accumulate_monthly_credit_sums(monthly_credit, wire, chunksize=50_000)

feat_avg_mo_credit = monthly_mean_from_dict(monthly_credit, "avg_monthly_credit_all")

# income_transaction_gap_score (individuals only)
feat_income_gap = kyc_ind[[
    "customer_id", "declared_monthly_income",
    "occupation_sector_id", "is_retired", "is_unemployed"
]].merge(feat_avg_mo_credit, on="customer_id", how="left")

feat_income_gap["avg_monthly_credit_all"] = feat_income_gap["avg_monthly_credit_all"].fillna(0.0)
feat_income_gap["income_transaction_gap_score"] = np.where(
    feat_income_gap["declared_monthly_income"] > 0,
    feat_income_gap["avg_monthly_credit_all"] / feat_income_gap["declared_monthly_income"],
    np.nan
)



In [25]:
# 5) Feature: cash_value_ratio (ABM cash activity / total tx amount)
#    Here "total tx amount" is sum(amount_cad) across all channels

def accumulate_total_amount(total_dict: dict, df: pd.DataFrame, chunksize: int):
    for chunk in _iter_chunks(df, chunksize):
        c = _prep_tx_chunk(chunk)
        g = c.groupby("customer_id")["amount_cad"].sum()
        for cust, amt in g.items():
            total_dict[cust] = total_dict.get(cust, 0.0) + float(amt)

total_amt = {}
accumulate_total_amount(total_amt, abm, CHUNK_ABM)
accumulate_total_amount(total_amt, card, CHUNK_CARD)
accumulate_total_amount(total_amt, cheque, CHUNK_MED)
accumulate_total_amount(total_amt, eft, CHUNK_BIG)
accumulate_total_amount(total_amt, emt, CHUNK_BIG)
accumulate_total_amount(total_amt, westernunion, 50_000)
accumulate_total_amount(total_amt, wire, 50_000)

total_amt_df = pd.DataFrame({"customer_id": list(total_amt.keys()), "total_tx_amount": list(total_amt.values())})

# cash total: ABM where cash_indicator==1 (both D/C count as cash usage)
cash_amt = {}
for chunk in _iter_chunks(abm, CHUNK_ABM):
    c = _prep_tx_chunk(chunk)
    c = c[c["cash_indicator"].eq(1)]
    g = c.groupby("customer_id")["amount_cad"].sum()
    for cust, amt in g.items():
        cash_amt[cust] = cash_amt.get(cust, 0.0) + float(amt)

cash_amt_df = pd.DataFrame({"customer_id": list(cash_amt.keys()), "cash_amount": list(cash_amt.values())})

feat_cash_ratio = total_amt_df.merge(cash_amt_df, on="customer_id", how="left")
feat_cash_ratio["cash_amount"] = feat_cash_ratio["cash_amount"].fillna(0.0)
feat_cash_ratio["cash_value_ratio"] = np.where(
    feat_cash_ratio["total_tx_amount"] > 0,
    feat_cash_ratio["cash_amount"] / feat_cash_ratio["total_tx_amount"],
    0.0
)
feat_cash_ratio = feat_cash_ratio[["customer_id", "cash_value_ratio"]]

In [26]:
# 7) Feature: spending_income_ratio (avg monthly CARD debits / declared income)
#    Compute avg monthly card debit spend via chunked monthly sums

card_monthly_debit = {}
for chunk in _iter_chunks(card, CHUNK_CARD):
    c = _prep_tx_chunk(chunk)
    c = c[c["debit_credit"].eq("D")]  # spend
    if c.empty:
        continue
    c["month"] = c["transaction_datetime"].dt.to_period("M").astype(str)
    g = c.groupby(["customer_id", "month"])["amount_cad"].sum()
    for (cust, m), s in g.items():
        card_monthly_debit[(cust, m)] = card_monthly_debit.get((cust, m), 0.0) + float(s)

feat_avg_mo_card_spend = monthly_mean_from_dict(card_monthly_debit, "avg_monthly_card_spend")

feat_spend_inc = kyc_ind[["customer_id", "declared_monthly_income"]].merge(
    feat_avg_mo_card_spend, on="customer_id", how="left"
)
feat_spend_inc["avg_monthly_card_spend"] = feat_spend_inc["avg_monthly_card_spend"].fillna(0.0)
feat_spend_inc["spending_income_ratio"] = np.where(
    feat_spend_inc["declared_monthly_income"] > 0,
    feat_spend_inc["avg_monthly_card_spend"] / feat_spend_inc["declared_monthly_income"],
    np.nan
)
feat_spend_inc = feat_spend_inc[["customer_id", "spending_income_ratio"]]

In [29]:
# 9) Feature: rapid_outflow_ratio_24h (computed for labeled customers only for speed)
#    - deposit proxy: ABM cash_indicator=1 & credit
#    - outflow: any debit across (CARD, EFT, EMT, CHEQUE, WU, WIRE, ABM debit)

def rapid_outflow_ratio_24h_for_customers(
    customers_list,
    abm_df, card_df, eft_df, emt_df, cheque_df, wu_df, wire_df,
    horizon_hours=24
) -> pd.DataFrame:

    cust_set = set(customers_list)
    H = np.timedelta64(horizon_hours, "h")  # IMPORTANT: numpy timedelta64

    # --- deposits: ABM cash deposits (credit) ---
    dep = abm_df[
        (abm_df["customer_id"].isin(cust_set)) &
        (abm_df["cash_indicator"].eq(1)) &
        (abm_df["debit_credit"].astype(str).str.upper().eq("C"))
    ][["customer_id", "transaction_datetime", "amount_cad"]].copy()

    dep["transaction_datetime"] = pd.to_datetime(dep["transaction_datetime"], errors="coerce")
    dep["amount_cad"] = pd.to_numeric(dep["amount_cad"], errors="coerce")
    dep = dep.dropna(subset=["customer_id", "transaction_datetime", "amount_cad"])
    dep = dep[dep["amount_cad"] > 0]

    # helper: get debits for a customer subset
    def _debits(df):
        x = df[df["customer_id"].isin(cust_set)][
            ["customer_id", "transaction_datetime", "amount_cad", "debit_credit"]
        ].copy()

        x["transaction_datetime"] = pd.to_datetime(x["transaction_datetime"], errors="coerce")
        x["amount_cad"] = pd.to_numeric(x["amount_cad"], errors="coerce")
        x["debit_credit"] = x["debit_credit"].astype(str).str.upper()

        x = x[x["debit_credit"].eq("D")].dropna(subset=["customer_id", "transaction_datetime", "amount_cad"])
        x = x[x["amount_cad"] > 0]
        return x[["customer_id", "transaction_datetime", "amount_cad"]]

    out = pd.concat([
        _debits(abm_df),
        _debits(card_df),
        _debits(eft_df),
        _debits(emt_df),
        _debits(cheque_df),
        _debits(wu_df),
        _debits(wire_df),
    ], ignore_index=True)

    # If no deposits at all, return zeros
    if dep.empty:
        return pd.DataFrame({"customer_id": list(cust_set), "rapid_outflow_ratio_24h": 0.0})

    dep = dep.sort_values(["customer_id", "transaction_datetime"])
    out = out.sort_values(["customer_id", "transaction_datetime"])

    rows = []

    # group once for speed
    out_groups = {cid: g for cid, g in out.groupby("customer_id", sort=False)}

    for cust, gdep in dep.groupby("customer_id", sort=False):
        gout = out_groups.get(cust)

        dep_total = float(gdep["amount_cad"].sum())
        if dep_total <= 0 or gout is None or gout.empty:
            rows.append((cust, 0.0))
            continue

        # Force BOTH arrays to numpy datetime64[ns]
        dep_times = gdep["transaction_datetime"].to_numpy(dtype="datetime64[ns]")
        out_times = gout["transaction_datetime"].to_numpy(dtype="datetime64[ns]")

        out_amt = gout["amount_cad"].to_numpy(dtype=float)
        out_cum = np.cumsum(out_amt)

        out_sum = 0.0
        for t in dep_times:
            # search window [t, t+H]
            left = np.searchsorted(out_times, t, side="left")
            right = np.searchsorted(out_times, t + H, side="right") - 1
            if right < left:
                continue
            out_sum += float(out_cum[right] - (out_cum[left - 1] if left > 0 else 0.0))

        rows.append((cust, out_sum / dep_total))

    feat_rapid = pd.DataFrame(rows, columns=["customer_id", "rapid_outflow_ratio_24h"])

    # customers with no deposits -> ratio 0
    missing = cust_set - set(feat_rapid["customer_id"])
    if missing:
        feat_rapid = pd.concat(
            [feat_rapid, pd.DataFrame({"customer_id": list(missing), "rapid_outflow_ratio_24h": 0.0})],
            ignore_index=True
        )

    return feat_rapid


In [33]:
# 10) Assemble final feature table + save to parquet

# 1) Compute rapid outflow (labeled customers only, as before)
labeled_customers = labels["customer_id"].unique()
feat_rapid = rapid_outflow_ratio_24h_for_customers(
    labeled_customers, abm, card, eft, emt, cheque, westernunion, wire,
    horizon_hours=24
)

# 2) Base customer universe (KYC universe) + labels
base = customers.merge(labels, on="customer_id", how="left")
base["label"] = pd.to_numeric(base["label"], errors="coerce")  # keep NaN for unlabeled

# 3) Assemble ONLY the final selected features
# Kept features:
# - cash_value_ratio
# - emt_velocity_7d_p95
# - income_transaction_gap_score (+ z + employment flags + occupation_sector_id)
# - spending_income_ratio
# - rapid_outflow_ratio_24h (only non-null for labeled customers => fill 0)
features = (
    base
    .merge(feat_cash_ratio, on="customer_id", how="left")
    .merge(feat_emt_vel, on="customer_id", how="left")
    .merge(
    feat_income_gap[[
        "customer_id",
        "income_transaction_gap_score",
        "is_retired",
        "is_unemployed",
        "occupation_sector_id",
    ]],
    on="customer_id",
    how="left"
)
    .merge(feat_spend_inc, on="customer_id", how="left")
    .merge(feat_rapid, on="customer_id", how="left")
)

# 4) Fill safe defaults
# (Ratios using declared income remain NaN if income missing; that's intentional)
for col in ["cash_value_ratio", "emt_velocity_monthly_mean", "rapid_outflow_ratio_24h"]:
    if col in features.columns:
        features[col] = features[col].fillna(0.0)

# 5) Save parquet
out_path = "Tony_AML_FE.parquet"
features.to_parquet(out_path, index=False)

print("Saved:", out_path)
print("Final feature columns:", list(features.columns))
features.head()

Saved: Tony_AML_FE.parquet
Final feature columns: ['customer_id', 'customer_type', 'label', 'cash_value_ratio', 'emt_velocity_monthly_mean', 'income_transaction_gap_score', 'is_retired', 'is_unemployed', 'occupation_sector_id', 'spending_income_ratio', 'rapid_outflow_ratio_24h']


,customer_id,customer_type,label,cash_value_ratio,emt_velocity_monthly_mean,income_transaction_gap_score,is_retired,is_unemployed,occupation_sector_id,spending_income_ratio,rapid_outflow_ratio_24h
0,SYNID0100000167,IND,NaN,0.000000,0.000000,0.807090,0.0,0.0,2.0,0.000000,0.0
1,SYNID0100000431,IND,NaN,0.183417,9.333333,NaN,0.0,0.0,18.0,NaN,0.0
2,SYNID0100000485,IND,0.0,0.037841,0.000000,1.460746,1.0,0.0,18.0,0.234281,0.0
3,SYNID0100000539,IND,NaN,0.000000,3.500000,0.016742,1.0,0.0,18.0,0.863900,0.0
4,SYNID0100000932,IND,NaN,0.012255,1.000000,17.062930,1.0,0.0,18.0,1.579105,0.0
